In [0]:

from pyspark.sql.functions import (col,sum,countDistinct,avg,round,year,month)

In [0]:
# KPI table
gold_kpi = (
    spark.table("fact_sales")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue"),
        round(sum("Profit"), 2).alias("TotalProfit"),
        sum("OrderQuantity").alias("TotalQuantity"),
        countDistinct("OrderNumber").alias("TotalOrders"),
        countDistinct("CustomerKey").alias("TotalCustomers"),
        round(avg("Revenue"), 2).alias("AverageRevenuePerLine")
    )
)

In [0]:
gold_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_kpi")

display(gold_kpi)

TotalRevenue,TotalProfit,TotalQuantity,TotalOrders,TotalCustomers,AverageRevenuePerLine
2.491458682E7,1.045771543E7,84174,25164,17416,444.54


In [0]:
# Monthly Revenue
gold_monthly_revenue = (
    spark.table("fact_sales")
    .withColumn("Year", year("OrderDate"))
    .withColumn("Month", month("OrderDate"))
    .groupBy("Year", "Month")
    .agg(
        round(sum("Revenue"), 2).alias("Revenue"),
        round(sum("Profit"), 2).alias("Profit"),
        sum("OrderQuantity").alias("Quantity"),
        countDistinct("OrderNumber").alias("Orders")
    )
    .orderBy("Year", "Month")
)

In [0]:
gold_monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_monthly_revenue")

display(gold_monthly_revenue)

Year,Month,Revenue,Profit,Quantity,Orders
2015,1,585312.65,235814.03,184,184
2015,2,532226.25,212186.69,165,165
2015,3,643436.1,259084.52,198,198
2015,4,653364.04,263031.34,204,204
2015,5,659325.9,266275.75,206,206
2015,6,669988.67,270067.51,212,212
2015,7,486115.01,196682.79,247,247
2015,8,536452.82,218355.47,278,278
2015,9,344062.87,140516.15,196,196
2015,10,404276.6,168581.76,223,223


In [0]:
#  Customer Summary
gold_customer_summary = (
    spark.table("fact_sales")
    .groupBy("CustomerKey")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue"),
        round(sum("Profit"), 2).alias("TotalProfit"),
        sum("OrderQuantity").alias("TotalQuantity"),
        countDistinct("OrderNumber").alias("TotalOrders"),
        round(avg("Revenue"), 2).alias("AverageRevenuePerLine")
    )
)

In [0]:
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_summary")

display(gold_customer_summary.limit(10))

CustomerKey,TotalRevenue,TotalProfit,TotalQuantity,TotalOrders,AverageRevenuePerLine
26782,699.1,285.95,1,1,699.1
18899,4118.26,1603.32,2,2,2059.13
14937,6407.0,2654.51,5,3,1281.4
14727,1585.67,707.15,5,2,396.42
11452,7945.45,3403.31,6,3,1589.09
29254,3578.27,1406.98,1,1,3578.27
14861,5743.15,2419.05,7,2,1148.63
18723,5279.26,2025.46,2,2,2639.63
11472,7940.8,3395.68,8,3,1323.47
28778,3578.27,1406.98,1,1,3578.27


In [0]:
# Product Summary
gold_product_summary = (
    spark.table("fact_sales")
    .groupBy("ProductKey")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue"),
        round(sum("Profit"), 2).alias("TotalProfit"),
        sum("OrderQuantity").alias("UnitsSold"),
        countDistinct("OrderNumber").alias("TotalOrders")
    )
)


In [0]:
gold_product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_product_summary")

display(gold_product_summary.limit(10))

ProductKey,TotalRevenue,TotalProfit,UnitsSold,TotalOrders
338,34954.91,14297.6,50,50
313,601149.36,236371.93,168,168
311,497379.53,195569.64,139,139
342,50335.07,20588.54,72,72
349,87749.74,38399.29,26,26
344,98599.71,43147.23,29,29
312,640510.33,251848.67,179,179
348,87749.74,38399.29,26,26
332,44742.28,18300.92,64,64
314,561788.39,220895.2,157,157


In [0]:
#  Category Summary
gold_category_summary = (
    spark.table("fact_sales")
    .join(
        spark.table("dim_product"),
        "ProductKey",
        "left"
    )
    .groupBy(
        "ProductCategoryKey",
        "CategoryName"
    )
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue"),
        round(sum("Profit"), 2).alias("TotalProfit"),
        sum("OrderQuantity").alias("UnitsSold"),
        countDistinct("ProductKey").alias("ProductsSold")
    )
)

In [0]:
gold_category_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_category_summary")

display(gold_category_summary)

ProductCategoryKey,CategoryName,TotalRevenue,TotalProfit,UnitsSold,ProductsSold
4,Accessories,906673.11,569760.06,57809,22
1,Bikes,2.36424951E7,9726168.27,13929,88
3,Clothing,365418.62,161787.1,12436,20


In [0]:
# Territory Summary
gold_territory_summary = (
    spark.table("fact_sales")
    .join(
        spark.table("dim_territory"),
        col("fact_sales.TerritoryKey") == col("dim_territory.SalesTerritoryKey"),
        "left"
    )
    .groupBy(
        "SalesTerritoryKey",
        "Region",
        "Country",
        "Continent"
    )
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue"),
        round(sum("Profit"), 2).alias("TotalProfit"),
        sum("OrderQuantity").alias("UnitsSold"),
        countDistinct("OrderNumber").alias("TotalOrders")
    )
)


In [0]:
gold_territory_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_territory_summary")

display(gold_territory_summary)

SalesTerritoryKey,Region,Country,Continent,TotalRevenue,TotalProfit,UnitsSold,TotalOrders
9,Australia,Australia,Pacific,7416456.2,3077022.81,17951,6060
3,Central,United States,North America,3143.06,1437.44,30,9
10,United Kingdom,United Kingdom,Europe,2902562.09,1214774.42,9694,2771
7,France,France,Europe,2362643.32,989346.06,7862,2315
1,Northwest,United States,North America,3095074.47,1314751.27,12513,3675
6,Canada,Canada,North America,1769245.81,757844.2,10894,3024
8,Germany,Germany,Europe,2524679.97,1054186.21,7950,2294
5,Southeast,United States,North America,11585.62,5133.29,49,14
2,Northeast,United States,North America,6401.57,2874.58,40,10
4,Southwest,United States,North America,4822794.7,2040345.17,17191,4992


In [0]:
#  Return Summary
gold_return_summary = (
    spark.table("fact_returns")
    .join(
        spark.table("dim_product"),
        "ProductKey",
        "left"
    )
    .groupBy(
        "ProductKey",
        "ProductName",
        "CategoryName"
    )
    .agg(
        sum("ReturnQuantity").alias("TotalReturnedQuantity")
    )
)

In [0]:
gold_return_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_return_summary")

display(gold_return_summary)

ProductKey,ProductName,CategoryName,TotalReturnedQuantity
313,"Road-150 Red, 52",Bikes,5
530,Touring Tire Tube,Accessories,45
566,"Touring-3000 Blue, 58",Bikes,2
324,"Road-650 Red, 62",Bikes,3
375,"Road-250 Black, 48",Bikes,8
535,LL Mountain Tire,Accessories,39
541,Touring Tire,Accessories,21
371,"Road-250 Red, 58",Bikes,11
483,Hitch Rack - 4-Bike,Accessories,8
588,"Mountain-400-W Silver, 40",Bikes,5


In [0]:
# Verifying all Gold tables
gold_tables = [
    "gold_kpi",
    "gold_monthly_revenue",
    "gold_customer_summary",
    "gold_product_summary",
    "gold_category_summary",
    "gold_territory_summary",
    "gold_return_summary"
]

for table in gold_tables:
    print(
        f"{table}: {spark.table(table).count()} rows"
    )

gold_kpi: 1 rows
gold_monthly_revenue: 30 rows
gold_customer_summary: 17416 rows
gold_product_summary: 130 rows
gold_category_summary: 3 rows
gold_territory_summary: 10 rows
gold_return_summary: 124 rows


In [0]:
spark.sql("""
SELECT COUNT(*) AS total_customers
FROM workspace.default.gold_customer_summary
""").show()

+---------------+
|total_customers|
+---------------+
|          17416|
+---------------+



In [0]:
#first initial run the total customers are 17416

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.bronze_customers
WHERE CustomerKey IN (11003, 11004)
""").show()

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|EducationLevel|  Occupation|HomeOwner|      ingestion_time|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|      11003|   MS.|  CHRISTY|     ZHU|1968-02-15|            S|     F|christy12@adventu...|    $70,000 |            0|     Bachelors|Professional|        N|2026-09-08 12:42:...|
|      11004|  MRS.|ELIZABETH| JOHNSON|1968-08-08|            S|     F|elizabeth5@advent...|    $80,000 |            5|     Bachelors|Professional|        Y|2026-09-08 12:42:...|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_customers
WHERE CustomerKey IN (11003, 11004)
""").show()

+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|CustomerKey|Prefix|FirstName|LastName| BirthDate|MaritalStatus|Gender|        EmailAddress|AnnualIncome|TotalChildren|EducationLevel|  Occupation|HomeOwner|      ingestion_time|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+-------------+--------------+------------+---------+--------------------+
|      11004|  MRS.|ELIZABETH| JOHNSON|1968-08-08|            S|     F|elizabeth5@advent...|    $80,000 |            5|     Bachelors|Professional|        Y|2026-09-08 12:42:...|
|      11003|   MS.|  CHRISTY|     ZHU|1968-02-15|            S|     F|christy12@adventu...|    $70,000 |            0|     Bachelors|Professional|        N|2026-09-08 12:42:...|
+-----------+------+---------+--------+----------+-------------+------+--------------------+------------+

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_customer_summary
WHERE CustomerKey IN (11003, 11004)
""").show()

+-----------+------------+-----------+-------------+-----------+---------------------+
|CustomerKey|TotalRevenue|TotalProfit|TotalQuantity|TotalOrders|AverageRevenuePerLine|
+-----------+------------+-----------+-------------+-----------+---------------------+
|      11004|     4568.08|    1926.73|            6|          2|               913.62|
|      11003|     4532.99|    1902.96|           11|          2|               566.62|
+-----------+------------+-----------+-------------+-----------+---------------------+

